# 🧬 Taller Módulo 10: Integración de transcriptómica en modelos metabólicos a escala genómica

### *Escherichia coli* ATCC 8739 creciendo en glicerol: del transcriptoma al diseño de una biofábrica de ácido succínico

## Introducción

El glicerol crudo es el principal subproducto de la industria del biodiésel (≈ 10 kg por cada 100 kg de biodiésel). Convertirlo en compuestos de valor agregado, como el **ácido succínico** —una molécula plataforma para polímeros biodegradables, solventes y alimentos—, es un ejemplo clásico de bioeconomía circular. *E. coli* es un buen candidato para esta biofábrica: tiene tres rutas hacia succinato, crece en glicerol y cuenta con herramientas de ingeniería genética y modelos metabólicos muy maduros.

Sin embargo, un modelo metabólico a escala genómica (GEM) describe **todo lo que la célula *podría* hacer**, no lo que *está haciendo* en una condición concreta. Dos células con el mismo genoma —por ejemplo, una cepa silvestre en un medio optimizado y una cepa sometida a evolución adaptativa en el laboratorio (ALE)— expresan genes distintos y, por lo tanto, usan rutas metabólicas distintas. La **integración de datos ómicos** permite construir **modelos específicos de contexto**: versiones del GEM que solo conservan las reacciones respaldadas por la expresión génica observada.

En este taller usted trabajará con datos reales de RNA-seq de *E. coli* ATCC 8739 cultivada aeróbicamente en glicerol (Tafur Rangel *et al.*, 2021, *Front. Genet.* 12:633073, [doi:10.3389/fgene.2021.633073](https://doi.org/10.3389/fgene.2021.633073)) y con el modelo **iEC1349_Crooks**, reconstruido para esta misma cepa. Con los resultados escribirá un **artículo científico**.

## Objetivos

Al terminar el taller usted podrá:

- Explorar y describir resultados de expresión diferencial (DESeq2) entre condiciones de cultivo.
- Conectar genes con reacciones mediante las reglas gen-proteína-reacción (GPR) de un GEM.
- Normalizar conteos de RNA-seq (TPM) y transformarlos en un nivel de expresión **por reacción**.
- Construir modelos específicos de contexto con el algoritmo **GIMME** implementado en COBRApy.
- Comparar distribuciones de flujo (pFBA) y rangos de flujo (FVA) entre condiciones.
- Predecir estrategias de *knockout* para sobreproducir ácido succínico y evaluar su acoplamiento al crecimiento.
- Generar figuras con calidad de publicación (PDF, ≤ 18 cm de ancho) y organizar los resultados para escribir un artículo.

## Casos de estudio: las tres condiciones experimentales

Todas las muestras se cultivaron en matraces de 250 mL con 50 mL de medio, aerobiosis, 37 °C y 200 rpm, por triplicado.

| Código | Muestras en los archivos | Cepa | Medio | Rol |
|:--|:--|:--|:--|:--|
| **REF** | `Expeleven1-3` (Exp 11) | Silvestre ATCC 8739 | Medio base de glicerol: 30 g/L glicerol, 5 g/L extracto de levadura, 2.5 g/L NaCl, MgSO₄, metales traza | Referencia |
| **OPT** | `Expthree1-3` (Exp 3) | Silvestre ATCC 8739 | Medio base + 1 g/L NH₄Cl, 6 g/L Na₂HPO₄, 3 g/L K₂HPO₄ (medio optimizado) | Efecto del medio |
| **ALE** | `Expteen1-3` (Exp 10) | Cepa evolucionada (ALE en altas concentraciones de glicerol) | Medio optimizado | Efecto de la evolución adaptativa |

Archivos de expresión diferencial (salida de DESeq2/SARTools):

| Archivo | Comparación | Identificadores |
|:--|:--|:--|
| `3vs11.txt` | OPT vs REF | nombre de gen (`Id`) + `locus_tag` |
| `10vs11.txt` | ALE vs REF | nombre de gen (`Id`) + `locus_tag` |
| `10vs3.txt` | ALE vs OPT | `locus_tag` (en la columna `Id`) |

Modelo metabólico:

| Característica | Valor |
|:--|:--|
| Modelo | iEC1349_Crooks (*E. coli* C, ATCC 8739) |
| Genes | 1.349 (identificadores `ECOLC_RS…`, los mismos del RNA-seq) |
| Reacciones | 2.756 |
| Metabolitos | 1.946 |
| Referencia original | Monk *et al.*, 2016, *Cell Syst.* / BiGG Models |

> **Nota:** el glicerol se suministró a 30 g/L. Antes de la ALE la cepa consumía ≈ 7 g/L; después ≈ 10 g/L; en el medio optimizado ≈ 30 g/L. Esta información le servirá para la discusión.

## 📝 Producto final: un artículo científico

Con los resultados de este cuaderno usted escribirá un artículo corto (formato IMRyD, estilo *Frontiers* / *PLOS*) con la siguiente estructura mínima:

1. **Título y resumen** (≤ 250 palabras).
2. **Introducción**: glicerol como sustrato, ácido succínico, GEMs e integración de ómicas.
3. **Métodos**: datos, normalización, modelo, GIMME, pFBA/FVA, diseño de cepas. *Al final del cuaderno se genera automáticamente un archivo `metodos_parametros.txt` con versiones y parámetros para citar.*
4. **Resultados**: organizados alrededor de las Figuras 1–7.
5. **Discusión**: interprete biológicamente, compare con la literatura y reconozca las limitaciones.
6. **Conclusiones y referencias.**

Todas las figuras se guardan en **PDF**, con un ancho máximo de **18 cm** (dos columnas) u **8.5 cm** (una columna), listas para insertar en el manuscrito.

> ⚠️ **Este taller es personal.** En el Paso 0.2 usted escribe su código de estudiante y el cuaderno le asigna parámetros propios (umbral de GIMME, fracción de crecimiento, nivel de oxígeno, un gen y una reacción para analizar). Sus resultados serán **distintos** a los de sus compañeros. Su artículo debe:
> - Reportar **sus** parámetros en Métodos.
> - Discutir explícitamente **su gen asignado** y **su reacción asignada**.
> - Incluir su **código de verificación** (se imprime al final del cuaderno) en la sección *Disponibilidad de datos*.
>
> El examen incluirá preguntas sobre **sus** resultados y **su** manuscrito.

## 🤔 Preguntas guía para este taller

Tenga en mente estas preguntas mientras trabaja:

- **¿Qué cambia en el transcriptoma cuando se optimiza el medio y cuando la cepa evoluciona?**
- **¿Esos cambios se concentran en el metabolismo o en otros procesos celulares?**
- **¿Cómo se traduce un nivel de expresión génica en una restricción sobre el flujo de una reacción?**
- **¿Un modelo específico de contexto predice un metabolismo distinto al del modelo genérico?**
- **¿Las estrategias de ingeniería metabólica dependen del contexto en el que se construyó el modelo?**

## ⚙️ Paso 0 — Configurar el entorno

En este taller **no** necesitamos Conda: todas las herramientas se instalan con `pip`. Ejecute esta celda primero.
⏱️ Puede tardar 1–2 minutos.

In [ ]:
# Instalar las herramientas y verificar el resultado
import os, sys, subprocess

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "cobra", "matplotlib", "pandas", "numpy", "scipy", "--quiet"],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f"❌ Falló pip install:\n{result.stderr}")

# Imports centralizados
import ast, json, glob, shutil, hashlib, random, itertools, time, zipfile, datetime, platform, urllib.request
from IPython.display import display

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
import cobra
from cobra.io import read_sbml_model, write_sbml_model
from cobra.flux_analysis import pfba, flux_variability_analysis, production_envelope

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

VERSIONES = {
    "Python": sys.version.split()[0],
    "cobra": cobra.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
}
print("📦 Versiones (cítelas en Métodos):")
for k, v in VERSIONES.items():
    print(f"  - {k}: {v}")
print(f"  - Google Colab: {'sí' if EN_COLAB else 'no detectado'}")

### 0.1 Carpetas de trabajo, estilo de figuras y funciones auxiliares

Todas las figuras de este taller cumplen las normas habituales de las revistas:

- **Ancho:** 8.5 cm (una columna) o 18 cm (dos columnas). **Nunca más de 18 cm.**
- **Formato:** PDF vectorial con fuentes incrustadas (`pdf.fonttype = 42`, texto editable).
- **Tipografía:** sans-serif de 7 pt (mínimo legible en impresión).
- **Colores:** cada condición tiene **siempre** el mismo color en todas las figuras (REF azul, OPT naranja, ALE verde azulado). La paleta fue verificada para personas con daltonismo; aun así, todas las figuras incluyen leyenda o etiquetas.

La función `guardar_figura()` comprueba el ancho antes de guardar: si una figura supera 18 cm, lanza un error.

In [ ]:
# ---------- Carpetas ----------
WORKDIR = os.path.abspath("taller_m10")
DATOS   = f"{WORKDIR}/datos"
RES     = f"{WORKDIR}/resultados"
FIG     = f"{RES}/figuras"
TAB     = f"{RES}/tablas"
MOD     = f"{RES}/modelos"
for d in (DATOS, FIG, TAB, MOD):
    os.makedirs(d, exist_ok=True)

# ---------- Estilo de figuras ----------
CM = 1 / 2.54                 # pulgadas por cm
ANCHO_1COL, ANCHO_2COL = 8.5, 18.0
ANCHO_MAX = 18.0

plt.rcParams.update({
    "pdf.fonttype": 42, "ps.fonttype": 42,          # fuentes incrustadas/editables
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7, "axes.titlesize": 8, "axes.labelsize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#52514e", "axes.labelcolor": "#0b0b0b",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "lines.linewidth": 1.2,
    "figure.dpi": 150, "savefig.dpi": 600,
})

# Un color fijo por condición (paleta validada para daltonismo)
COLOR = {"REF": "#2a78d6", "OPT": "#eb6834", "ALE": "#1baf7a", "Base": "#898781"}
NOMBRE = {"REF": "REF (medio base)", "OPT": "OPT (medio optimizado)",
          "ALE": "ALE (cepa evolucionada)", "Base": "Modelo genérico"}
GRIS = "#c3c2b7"
# Mapa divergente azul–gris–rojo para log2FC y flujos con signo
DIVERGENTE = LinearSegmentedColormap.from_list(
    "div", ["#104281", "#3987e5", "#cde2fb", "#f0efec", "#f7c6c5", "#e34948", "#8c1d1d"])

def nueva_figura(columnas=2, alto_cm=8.0, **kwargs):
    # Crea una figura de 1 o 2 columnas (8.5 o 18 cm de ancho)
    ancho = ANCHO_2COL if columnas == 2 else ANCHO_1COL
    return plt.figure(figsize=(ancho * CM, alto_cm * CM), layout="constrained", **kwargs)

def letra_panel(ax, letra):
    ax.text(-0.02, 1.02, letra, transform=ax.transAxes, fontsize=10,
            fontweight="bold", va="bottom", ha="right")

def guardar_figura(fig, nombre):
    # Verifica el ancho (≤ 18 cm) y guarda la figura en PDF
    ancho_cm = fig.get_figwidth() / CM
    if ancho_cm > ANCHO_MAX + 0.01:
        raise ValueError(f"❌ La figura mide {ancho_cm:.1f} cm de ancho (máximo {ANCHO_MAX} cm)")
    ruta = f"{FIG}/{nombre}.pdf"
    meta = {"Author": globals().get("CODIGO_ESTUDIANTE", ""),
            "Subject": f"Taller M10 | verificación {globals().get('CODIGO_VERIFICACION', '')}"}
    fig.savefig(ruta, metadata=meta)  # sin bbox_inches='tight' para conservar el tamaño exacto
    if not os.path.exists(ruta) or os.path.getsize(ruta) == 0:
        raise RuntimeError(f"❌ No se guardó {ruta}")
    print(f"✅ {nombre}.pdf  ({ancho_cm:.1f} × {fig.get_figheight()/CM:.1f} cm)")
    plt.show()

def guardar_tabla(df, nombre, index=False):
    ruta = f"{TAB}/{nombre}.csv"
    df.to_csv(ruta, index=index)
    print(f"✅ Tabla guardada: {ruta}")

# Registro de parámetros para la sección de Métodos
PARAMETROS = {}
print(f"✅ Carpeta de trabajo: {WORKDIR}")

### 0.2 Su asignación personal

Escriba su **código de estudiante** (tal como aparece en el sistema de la universidad) y ejecute la celda. El cuaderno le asignará, de forma reproducible, los parámetros con los que trabajará durante todo el taller. Si vuelve a ejecutar con el mismo código, obtendrá siempre la misma asignación.

> No cambie estos valores manualmente: su docente puede reproducir su asignación y sus resultados a partir de su código.

In [ ]:
CODIGO_ESTUDIANTE = ""      # ✏️ escriba aquí su código, por ejemplo: "202112345"

CODIGO_ESTUDIANTE = str(CODIGO_ESTUDIANTE).strip().upper()
if not CODIGO_ESTUDIANTE:
    raise ValueError("❌ Escriba su código de estudiante en CODIGO_ESTUDIANTE antes de continuar.")

OPCIONES = {
    "gimme_percentil": [15, 20, 25, 30, 35],
    "gimme_fraccion":  [0.80, 0.85, 0.90, 0.95],
    "o2_limitado":     [2, 3, 4, 5, 6],
    "gen_asignado":    ["glpK", "glpD", "glpF", "glpT", "gldA", "dhaK", "tpiA", "glpX", "ppc", "pck",
                        "maeB", "aceE", "pflB", "gltA", "sdhA", "fumA", "frdA", "aceA", "pta", "poxB"],
    "reaccion_asignada": ["GLYK", "GLYCDx", "DHAPT", "G3PD2", "PYK", "PDH", "PPC", "CS",
                          "ICDHyr", "AKGDH", "SUCOAS", "SUCDi", "FUM", "MDH"],
}

def asignar(codigo):
    # Asignación determinista a partir del código (la misma para el estudiante y el docente)
    rng = random.Random(int(hashlib.sha256(f"M10-taller|{codigo}".encode()).hexdigest(), 16))
    return {k: rng.choice(v) for k, v in OPCIONES.items()}

def codigo_verificacion(codigo, asignacion):
    huella = hashlib.sha256(f"M10-verif|{codigo}|{json.dumps(asignacion, sort_keys=True)}".encode()).hexdigest()
    return f"{huella[:4]}-{huella[4:8]}".upper()

ASIGNACION = asignar(CODIGO_ESTUDIANTE)
CODIGO_VERIFICACION = codigo_verificacion(CODIGO_ESTUDIANTE, ASIGNACION)
PARAMETROS["estudiante"] = {"codigo": CODIGO_ESTUDIANTE, "asignacion": ASIGNACION,
                            "codigo_verificacion": CODIGO_VERIFICACION}

print(f"👤 Estudiante: {CODIGO_ESTUDIANTE}")
print(f"   Umbral de GIMME (percentil):        {ASIGNACION['gimme_percentil']}")
print(f"   Fracción de crecimiento en GIMME:   {ASIGNACION['gimme_fraccion']:.2f}")
print(f"   Captación de O2 en escenario limitado: {ASIGNACION['o2_limitado']} mmol/gDW/h")
print(f"   Gen asignado:      {ASIGNACION['gen_asignado']}")
print(f"   Reacción asignada: {ASIGNACION['reaccion_asignada']}")
print(f"\n🔑 Código de verificación: {CODIGO_VERIFICACION}  (inclúyalo en su artículo)")

## 📥 Paso 1 — Descargar los datos

Los archivos están en el repositorio del curso en GitHub. Si la descarga falla (por ejemplo, sin conexión), la celda le pedirá subirlos manualmente desde su computador.

In [ ]:
REPO = "https://raw.githubusercontent.com/ae-tafur/bioinformatics/main/10-metabolic-models/exercises/data"
ARCHIVOS = {
    "iEC1349_Crooks.xml": f"{REPO}/e_coli/models/iEC1349_Crooks.xml",   # modelo metabólico
    "iEC1349.faa":        f"{REPO}/genomes/iEC1349.faa",                # proteoma (longitud de genes)
    "3vs11.txt":          f"{REPO}/e_coli/data/3vs11.txt",              # OPT vs REF
    "10vs11.txt":         f"{REPO}/e_coli/data/10vs11.txt",             # ALE vs REF
    "10vs3.txt":          f"{REPO}/e_coli/data/10vs3.txt",              # ALE vs OPT
}

faltantes = []
for nombre, url in ARCHIVOS.items():
    destino = f"{DATOS}/{nombre}"
    if not os.path.exists(destino):
        try:
            urllib.request.urlretrieve(url, destino)
        except Exception as e:
            print(f"⚠️ No se pudo descargar {nombre}: {e}")
            faltantes.append(nombre)
            continue
    print(f"✅ {nombre}: {os.path.getsize(destino)/1e6:.2f} MB")

if faltantes:
    if not EN_COLAB:
        raise FileNotFoundError(f"❌ Copie manualmente a {DATOS}: {faltantes}")
    from google.colab import files
    print(f"\n⬆️ Suba estos archivos: {faltantes}")
    for nombre, contenido in files.upload().items():
        with open(f"{DATOS}/{nombre}", "wb") as f:
            f.write(contenido)
    falta = [n for n in faltantes if not os.path.exists(f"{DATOS}/{n}")]
    if falta:
        raise FileNotFoundError(f"❌ Siguen faltando: {falta}")
    print("✅ Archivos subidos")

## 📊 Parte 1 — Exploración del transcriptoma

### 1.1 Cargar las tablas de expresión diferencial

Cada tabla contiene los conteos crudos, los conteos normalizados por DESeq2, el *log2 fold change* (log2FC), el valor *p* y el valor *p* ajustado por FDR (`padj`).

En el artículo original se definió como gen diferencialmente expresado (DEG) aquel con **|log2FC| ≥ 2** y **padj < 0.05**. Usaremos el mismo criterio para poder comparar.

**Preguntas antes de ejecutar:**
- ¿Por qué se usa el valor *p* ajustado y no el valor *p* crudo?
- ¿Qué significa biológicamente un log2FC = 2? ¿Y uno de −2?

In [ ]:
LFC_UMBRAL  = 2.0     # |log2FC| mínimo (criterio del artículo original)
PADJ_UMBRAL = 0.05    # FDR máximo
PARAMETROS["DEG"] = {"abs_log2FC": LFC_UMBRAL, "padj": PADJ_UMBRAL}

t_opt = pd.read_csv(f"{DATOS}/3vs11.txt")    # OPT vs REF
t_ale = pd.read_csv(f"{DATOS}/10vs11.txt")   # ALE vs REF
t_ao  = pd.read_csv(f"{DATOS}/10vs3.txt")    # ALE vs OPT
t_ao  = t_ao.rename(columns={"Id": "locus_tag"})

# Diccionario locus_tag -> nombre del gen (10vs3 solo trae locus_tag)
nombre_gen = pd.concat([t_opt.set_index("locus_tag")["Id"],
                        t_ale.set_index("locus_tag")["Id"]]).groupby(level=0).first()
t_ao["Id"] = t_ao["locus_tag"].map(nombre_gen).fillna(t_ao["locus_tag"])

COMPARACIONES = {"OPT vs REF": t_opt, "ALE vs REF": t_ale, "ALE vs OPT": t_ao}

def clasificar(df):
    sig = (df["padj"] < PADJ_UMBRAL) & (df["log2FoldChange"].abs() >= LFC_UMBRAL)
    return np.where(sig & (df["log2FoldChange"] > 0), "Sube",
           np.where(sig & (df["log2FoldChange"] < 0), "Baja", "NS"))

filas = []
for comp, df in COMPARACIONES.items():
    df["clase"] = clasificar(df)
    filas.append({"Comparación": comp, "Genes evaluados": len(df),
                  "DEG suben": int((df.clase == "Sube").sum()),
                  "DEG bajan": int((df.clase == "Baja").sum()),
                  "DEG totales": int((df.clase != "NS").sum())})
resumen_deg = pd.DataFrame(filas)
display(resumen_deg)
guardar_tabla(resumen_deg, "T1_resumen_DEG")

**Análisis:**
- Compare sus números con los del artículo original (OPT: 478 DEG, 256 suben / 222 bajan; ALE: 431 DEG, 208 suben / 223 bajan). ¿Coinciden?
- ¿Por qué la comparación ALE vs OPT evalúa más genes que las otras dos? (Pista: revise cómo se filtraron los genes con pocos conteos.)
- ¿Cuál de las dos intervenciones (cambiar el medio o evolucionar la cepa) perturba más el transcriptoma?

### 1.2 Figura 1 — Panorama de la expresión diferencial

Construiremos una figura de tres paneles:
- **A y B:** gráficos de volcán (*volcano plots*) de OPT vs REF y ALE vs REF, resaltando los genes del metabolismo del glicerol.
- **C:** comparación directa de los log2FC de ambas condiciones frente a REF. Si la evolución adaptativa y el medio optimizado producen respuestas parecidas, los puntos caerán sobre la diagonal.

In [ ]:
GENES_GLICEROL = ["glpF", "glpK", "glpD", "glpA", "glpB", "glpC", "glpT", "glpQ", "glpX", "gldA", "dhaK", "dhaL", "dhaM"]

def volcan(ax, df, titulo, color):
    y = -np.log10(df["padj"].clip(lower=1e-300))
    ns = df["clase"] == "NS"
    ax.scatter(df.loc[ns, "log2FoldChange"], y[ns], s=2, color=GRIS, lw=0, rasterized=True, label="NS")
    ax.scatter(df.loc[~ns, "log2FoldChange"], y[~ns], s=3, color=color, lw=0, rasterized=True,
               label=f"DEG (n = {(~ns).sum()})")
    gl = df["Id"].isin(GENES_GLICEROL)
    ax.scatter(df.loc[gl, "log2FoldChange"], y[gl], s=10, facecolor="none", edgecolor="#0b0b0b", lw=0.6,
               label="Metabolismo de glicerol")
    for _, r in df[gl & (df["log2FoldChange"].abs() > 3)].iterrows():
        ax.annotate(r["Id"], (r["log2FoldChange"], -np.log10(max(r["padj"], 1e-300))),
                    fontsize=5, xytext=(2, 2), textcoords="offset points", style="italic")
    for x in (-LFC_UMBRAL, LFC_UMBRAL):
        ax.axvline(x, color="#898781", lw=0.5, ls="--")
    ax.axhline(-np.log10(PADJ_UMBRAL), color="#898781", lw=0.5, ls="--")
    ax.set_xlabel("log$_2$ fold change")
    ax.set_ylabel("−log$_{10}$ (padj)")
    ax.set_title(titulo)
    ax.legend(loc="upper left", handletextpad=0.2, borderaxespad=0.2, markerscale=1.5)

# Máximo dos paneles por fila; el tercero va centrado abajo (9 cm de ancho)
fig = nueva_figura(columnas=2, alto_cm=13.0)
gs = fig.add_gridspec(2, 4)
axs = [fig.add_subplot(gs[0, 0:2]), fig.add_subplot(gs[0, 2:4]), fig.add_subplot(gs[1, 1:3])]
volcan(axs[0], t_opt, "OPT vs REF", COLOR["OPT"])
volcan(axs[1], t_ale, "ALE vs REF", COLOR["ALE"])

# Panel C: log2FC OPT vs log2FC ALE (genes presentes en ambas comparaciones)
comb = t_opt.set_index("locus_tag")[["Id", "log2FoldChange", "clase"]].join(
       t_ale.set_index("locus_tag")[["log2FoldChange", "clase"]], lsuffix="_opt", rsuffix="_ale", how="inner")
ax = axs[2]
ambos = (comb.clase_opt != "NS") & (comb.clase_ale != "NS")
ax.scatter(comb.log2FoldChange_opt, comb.log2FoldChange_ale, s=2, color=GRIS, lw=0, rasterized=True, label="Todos")
ax.scatter(comb.loc[ambos, "log2FoldChange_opt"], comb.loc[ambos, "log2FoldChange_ale"], s=3,
           color="#6250d6", lw=0, rasterized=True, label=f"DEG en ambas (n = {ambos.sum()})")
lim = np.nanmax(np.abs(comb[["log2FoldChange_opt", "log2FoldChange_ale"]].values)) * 1.05
ax.plot([-lim, lim], [-lim, lim], color="#898781", lw=0.5, ls="--")
r_p, p_p = stats.pearsonr(comb.log2FoldChange_opt, comb.log2FoldChange_ale)
ax.text(0.97, 0.04, f"r de Pearson = {r_p:.2f}", transform=ax.transAxes, ha="right", fontsize=6)
ax.set_xlabel("log$_2$FC OPT vs REF"); ax.set_ylabel("log$_2$FC ALE vs REF")
ax.set_title("Respuesta compartida"); ax.legend(loc="upper left", markerscale=1.5)
for a, l in zip(axs, "ABC"):
    letra_panel(a, l)
guardar_figura(fig, "Figura1_expresion_diferencial")
print(f"Correlación de log2FC entre OPT y ALE: r = {r_p:.3f} (p = {p_p:.1e}), n = {len(comb)} genes")

**Análisis — Figura 1:**
- ¿Qué genes del metabolismo del glicerol aparecen entre los más inducidos? ¿Tiene sentido si el medio optimizado permite consumir ≈ 4 veces más glicerol?
- ¿Qué indica una correlación positiva y alta en el panel C? ¿Qué genes se alejan de la diagonal y qué podrían significar?
- Busque en el volcán el gen con el log2FC más negativo en ambas comparaciones (pista: `phoE`). ¿Por qué un medio suplementado con fosfato reprimiría una porina inducida por limitación de fosfato?

### 1.3 Figura 2 — Genes del metabolismo central relacionados con glicerol y succinato

La Figura 1 de Tafur Rangel *et al.* (2021) muestra las rutas de glicerol a succinato. Aquí miramos la expresión de los genes de esas rutas en las tres comparaciones.

**Preguntas antes de ejecutar:**
- ¿Por qué ruta espera que entre el glicerol en aerobiosis: glicerol quinasa (`glpK`) o glicerol deshidrogenasa (`gldA`)?
- ¿Qué genes esperaría inducidos si la célula produjera más succinato por la rama reductiva (fumarato reductasa)?

In [ ]:
PANEL_GENES = {
    "Glicerol":      ["glpF", "glpK", "glpD", "glpA", "glpB", "glpC", "gldA", "dhaK", "dhaL", "dhaM", "glpT", "glpQ"],
    "Glucólisis": ["tpiA", "fbaA", "fbaB", "glpX", "fsaA", "fsaB"],
    "PEP/piruvato":        ["ppc", "pck", "maeA", "maeB", "aceE", "aceF", "lpd", "pflB"],
    "TCA y glioxilato":         ["gltA", "icd", "sucA", "sucB", "sucC", "sucD", "sdhA", "sdhB", "sdhC", "sdhD",
                                 "fumA", "fumB", "fumC", "mdh", "aceA", "aceB"],
    "Reductiva":           ["frdA", "frdB", "frdC", "frdD"],
    "Acetato/ferm.":   ["pta", "ackA", "poxB", "ldhA", "adhE"],
    "C4":         ["dcuA", "dcuB", "dcuC", "dctA"],
}
orden, grupo = [], []
for g, genes in PANEL_GENES.items():
    orden += genes; grupo += [g] * len(genes)

def por_nombre(df):
    # Algunos nombres de gen se repiten (parálogos sin nombre único): se conserva la copia más expresada
    return df.sort_values("baseMean", ascending=False).drop_duplicates("Id").set_index("Id")
lfc = pd.DataFrame({comp: por_nombre(df)["log2FoldChange"].reindex(orden) for comp, df in COMPARACIONES.items()})
padj = pd.DataFrame({comp: por_nombre(df)["padj"].reindex(orden) for comp, df in COMPARACIONES.items()})
tabla_panel = lfc.round(2).join(padj.add_suffix(" padj"))
tabla_panel.insert(0, "Ruta", grupo)
guardar_tabla(tabla_panel.reset_index().rename(columns={"index": "Gen"}), "T2_genes_rutas_centrales")

fig = nueva_figura(columnas=2, alto_cm=6.0)
ax = fig.subplots()
lim = np.ceil(np.nanmax(np.abs(lfc.values)))
im = ax.imshow(lfc.T.values, aspect="auto", cmap=DIVERGENTE, norm=TwoSlopeNorm(0, -lim, lim))
ax.set_yticks(range(3), lfc.columns)
ax.set_xticks(range(len(orden)), orden, rotation=90, style="italic")
# Asteriscos: significativo según el criterio del taller
for i, comp in enumerate(lfc.columns):
    for j, g in enumerate(orden):
        if padj.loc[g, comp] < PADJ_UMBRAL and abs(lfc.loc[g, comp]) >= 1:
            ax.text(j, i, "*", ha="center", va="center", fontsize=6,
                    color="white" if abs(lfc.loc[g, comp]) > lim * 0.6 else "#0b0b0b")
# Separadores y nombres de ruta
pos = 0
for g, genes in PANEL_GENES.items():
    if pos > 0:
        ax.axvline(pos - 0.5, color="white", lw=1.5)
    ax.text(pos + len(genes) / 2 - 0.5, -0.75, g, ha="center", va="bottom", fontsize=5.5)
    pos += len(genes)
ax.spines[:].set_visible(False); ax.tick_params(length=0)
cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.01)
cb.set_label("log$_2$FC"); cb.outline.set_visible(False)
ax.set_title(" ", pad=12)
guardar_figura(fig, "Figura2_genes_rutas_centrales")
print("* padj < 0.05 y |log2FC| ≥ 1. Gris = gen sin datos (filtrado por bajos conteos).")

**Análisis — Figura 2:**
- Compare `glpK`/`glpD` con `gldA`/`dhaKLM`. ¿Qué ruta de entrada de glicerol favorecen OPT y ALE?
- ¿Cómo cambia la expresión de fumarasa (`fumA/B/C`) y fumarato reductasa (`frdABCD`) en ALE frente a OPT? Relaciónelo con la producción de succinato.
- `pta` está inducido en ambas condiciones. ¿Qué implicación tiene para la producción de acetato, un subproducto indeseado?
- **Limitación:** un log2FC alto no implica un flujo alto. ¿Qué otros niveles de regulación (post-transcripcional, alostérica, cinética) quedan fuera de este análisis?

### 1.4 🔎 Su gen asignado

La siguiente celda resume su gen asignado en las tres comparaciones y le muestra qué reacciones del modelo cataliza. **Debe discutirlo en su artículo**: ¿qué hace?, ¿cómo cambia su expresión?, ¿es coherente con el fenotipo (consumo de glicerol, producción de succinato)?

In [ ]:
GEN = ASIGNACION["gen_asignado"]
filas = []
for comp, df in COMPARACIONES.items():
    fila = df[df["Id"] == GEN]
    if len(fila):
        f = fila.sort_values("baseMean", ascending=False).iloc[0]
        filas.append({"Comparación": comp, "locus_tag": f["locus_tag"], "baseMean": round(f["baseMean"], 1),
                      "log2FC": round(f["log2FoldChange"], 3), "padj": f"{f['padj']:.2e}", "Clase": f["clase"]})
    else:
        filas.append({"Comparación": comp, "locus_tag": "—", "baseMean": np.nan, "log2FC": np.nan,
                      "padj": "sin datos", "Clase": "—"})
gen_tabla = pd.DataFrame(filas)
print(f"🔎 Gen asignado: {GEN}")
display(gen_tabla)
guardar_tabla(gen_tabla, f"P1_gen_asignado_{GEN}")
GEN_LOCUS = next((x for x in gen_tabla["locus_tag"] if x != "—"), None)

## 🔗 Parte 2 — Del gen a la reacción

### 2.1 Cargar el modelo y definir el medio de glicerol

Simularemos un medio mínimo aerobio con glicerol como única fuente de carbono. Como en el artículo original, la captación de glicerol se limita a **13.3 mmol/gDW/h**.

> **Supuesto importante:** el medio real contiene extracto de levadura (aminoácidos, vitaminas, nucleótidos). Representarlo con un medio mínimo es una simplificación que usted debe discutir en el artículo.

In [ ]:
modelo = read_sbml_model(f"{DATOS}/iEC1349_Crooks.xml")
BIOMASA = "BIOMASS_Ec_iJO1366_core_53p95M"
GLICEROL_UPTAKE = 13.3    # mmol/gDW/h (Tafur Rangel et al., 2021)

modelo.objective = BIOMASA
modelo.reactions.EX_glc__D_e.lower_bound = 0                 # sin glucosa
modelo.reactions.EX_glyc_e.lower_bound = -GLICEROL_UPTAKE    # glicerol como fuente de carbono
modelo.reactions.EX_o2_e.lower_bound = -1000                 # aerobiosis

PARAMETROS["modelo"] = {"id": modelo.id, "reacciones": len(modelo.reactions),
                        "metabolitos": len(modelo.metabolites), "genes": len(modelo.genes),
                        "objetivo": BIOMASA, "glicerol_uptake_mmol_gDW_h": GLICEROL_UPTAKE,
                        "solver": modelo.solver.interface.__name__.split(".")[-1]}

sol_base = modelo.optimize()
print("=" * 55)
print(f"Modelo: {modelo.id} | {len(modelo.reactions)} reacciones, "
      f"{len(modelo.metabolites)} metabolitos, {len(modelo.genes)} genes")
print(f"Crecimiento máximo en glicerol: {sol_base.objective_value:.3f} 1/h")
print(f"Rendimiento de biomasa: {sol_base.objective_value / (GLICEROL_UPTAKE * 0.09209):.3f} gDW/g glicerol")
print("=" * 55)
modelo.summary()

**Análisis:**
- ¿Qué productos secreta el modelo genérico? ¿Secreta succinato?
- Calcule el tiempo de duplicación correspondiente a la tasa de crecimiento predicha. ¿Es realista para *E. coli* en glicerol (≈ 0.4–0.6 1/h en medio mínimo)?

### 2.2 ¿Qué fracción del transcriptoma está representada en el modelo?

No todos los genes del genoma participan en el metabolismo. Una prueba exacta de Fisher nos dice si los DEG están **enriquecidos** (o empobrecidos) en genes metabólicos.

In [ ]:
genes_modelo = {g.id for g in modelo.genes}
filas = []
for comp, df in COMPARACIONES.items():
    en_mod = df["locus_tag"].isin(genes_modelo)
    deg = df["clase"] != "NS"
    tabla = [[(en_mod & deg).sum(), (en_mod & ~deg).sum()],
             [(~en_mod & deg).sum(), (~en_mod & ~deg).sum()]]
    odds, p = stats.fisher_exact(tabla)
    filas.append({"Comparación": comp, "Genes con datos": len(df), "Genes en el modelo": int(en_mod.sum()),
                  "DEG metabólicos": int((en_mod & deg).sum()), "DEG no metabólicos": int((~en_mod & deg).sum()),
                  "% DEG metabólicos": round(100 * (en_mod & deg).sum() / max(deg.sum(), 1), 1),
                  "Odds ratio": round(odds, 2), "p (Fisher)": f"{p:.2e}"})
cobertura = pd.DataFrame(filas)
display(cobertura)
guardar_tabla(cobertura, "T3_cobertura_modelo_enriquecimiento")

**Análisis:**
- ¿Los cambios transcripcionales se concentran en el metabolismo (odds ratio > 1) o fuera de él?
- ¿Qué procesos no metabólicos podrían explicar el resto de DEG? (Pista: estrés ácido `hdeAB`, porinas, movilidad.) Puede complementar con un análisis de GO como el de la Figura 3 del artículo original.

### 2.3 Normalización TPM y expresión por reacción

Los conteos normalizados por DESeq2 sirven para comparar **un mismo gen entre muestras**, pero no **genes distintos dentro de una muestra**, porque un gen largo acumula más lecturas. Para decidir si una reacción está "poco expresada" necesitamos comparar genes entre sí, así que usaremos **TPM** (*transcripts per million*), que corrige por la longitud del gen. La longitud se obtiene del proteoma (`iEC1349.faa`): longitud del gen ≈ 3 × (aminoácidos + 1).

Luego traducimos la expresión de genes a expresión de **reacciones** usando la regla GPR:
- **AND** (complejo enzimático, todas las subunidades son necesarias) → **mínimo**.
- **OR** (isoenzimas, cualquiera basta) → **máximo**.

**Preguntas:**
- ¿Por qué usamos el mínimo para un complejo y el máximo para isoenzimas?
- ¿Qué pasa con las reacciones sin genes asociados (difusión, intercambios)?

In [ ]:
# ---------- Matriz de conteos crudos (9 muestras) ----------
# REF aparece en dos archivos (con los mismos conteos crudos); OPT y ALE están completos en 10vs3.
ref_cols = [f"Expeleven{i}" for i in (1, 2, 3)]
ref = pd.concat([t_opt.set_index("locus_tag")[ref_cols], t_ale.set_index("locus_tag")[ref_cols]]).groupby(level=0).first()
otros = t_ao.set_index("locus_tag")[[f"Expthree{i}" for i in (1, 2, 3)] + [f"Expteen{i}" for i in (1, 2, 3)]]
conteos = otros.join(ref, how="left")
n_sin_ref = conteos[ref_cols].isna().any(axis=1).sum()
conteos = conteos.fillna(0)   # genes filtrados en REF por < 100 conteos promedio → expresión baja
conteos.columns = ["OPT_1", "OPT_2", "OPT_3", "ALE_1", "ALE_2", "ALE_3", "REF_1", "REF_2", "REF_3"]
conteos = conteos[["REF_1", "REF_2", "REF_3", "OPT_1", "OPT_2", "OPT_3", "ALE_1", "ALE_2", "ALE_3"]]
print(f"Matriz de conteos: {conteos.shape[0]} genes × {conteos.shape[1]} muestras "
      f"({n_sin_ref} genes sin conteos en REF se tratan como 0)")

# ---------- Longitud de genes a partir del proteoma ----------
longitud = {}
with open(f"{DATOS}/iEC1349.faa") as f:
    gid = None
    for linea in f:
        linea = linea.strip()
        if linea.startswith(">"):
            gid = linea[1:].split()[0].replace("G_", "")
            longitud[gid] = 0
        elif gid:
            longitud[gid] += len(linea)
long_pb = pd.Series({g: 3 * (aa + 1) for g, aa in longitud.items()}).reindex(conteos.index)
print(f"Genes con longitud conocida: {long_pb.notna().sum()} de {len(long_pb)} (al resto se le asigna la mediana)")
long_pb = long_pb.fillna(long_pb.median())

# ---------- TPM ----------
rpk = conteos.div(long_pb / 1000, axis=0)
tpm = rpk.div(rpk.sum(axis=0), axis=1) * 1e6
CONDICIONES = ["REF", "OPT", "ALE"]
tpm_media = pd.DataFrame({c: tpm[[f"{c}_{i}" for i in (1, 2, 3)]].mean(axis=1) for c in CONDICIONES})
log_tpm = np.log2(tpm_media + 1)
guardar_tabla(tpm.round(3).reset_index().rename(columns={"index": "locus_tag"}), "S1_TPM_por_muestra")

# Correlación entre réplicas (control de calidad)
corr = np.log2(tpm + 1).corr(method="spearman")
print("\nCorrelación de Spearman mínima entre réplicas de una misma condición:")
for c in CONDICIONES:
    cols = [f"{c}_{i}" for i in (1, 2, 3)]
    print(f"  {c}: {corr.loc[cols, cols].values[np.triu_indices(3, 1)].min():.3f}")

In [ ]:
def expresion_reaccion(rxn, expr, y=min, o=max):
    # Evalúa la regla GPR: AND -> mínimo, OR -> máximo. Devuelve NaN si no hay genes o datos.
    if not rxn.genes:
        return np.nan
    def evaluar(nodo):
        if isinstance(nodo, ast.Name):
            return expr.get(nodo.id, np.nan)
        if isinstance(nodo, ast.BoolOp):
            vals = [v for v in (evaluar(x) for x in nodo.values) if not np.isnan(v)]
            if not vals:
                return np.nan
            return y(vals) if isinstance(nodo.op, ast.And) else o(vals)
        if isinstance(nodo, ast.Expression):
            return evaluar(nodo.body)
        raise ValueError(f"Nodo GPR no esperado: {type(nodo)}")
    return evaluar(rxn.gpr.body)

expr_rxn = pd.DataFrame({c: {r.id: expresion_reaccion(r, log_tpm[c].to_dict()) for r in modelo.reactions}
                         for c in CONDICIONES})
print(f"Reacciones con expresión asignada: {expr_rxn['REF'].notna().sum()} de {len(expr_rxn)}")
display(expr_rxn.loc[["GLYK", "G3PD5", "G3PD2", "GLYCDx", "DHAPT", "PDH", "PFL", "FUM", "FRD2", "SUCDi", "PTAr", "ACKr"]].round(2))
guardar_tabla(expr_rxn.round(3).reset_index().rename(columns={"index": "reaccion"}), "S2_expresion_por_reaccion")

# Reacciones catalizadas por su gen asignado
if GEN_LOCUS and GEN_LOCUS in modelo.genes:
    rx_gen = [r.id for r in modelo.genes.get_by_id(GEN_LOCUS).reactions]
    print(f"\n🔎 {GEN} ({GEN_LOCUS}) participa en {len(rx_gen)} reacción(es) del modelo:")
    for rid in rx_gen:
        r = modelo.reactions.get_by_id(rid)
        print(f"   {rid:<12} {r.name[:45]:<45} GPR: {r.gene_reaction_rule[:60]}")
    display(expr_rxn.loc[rx_gen].round(2))
else:
    print(f"\n🔎 {GEN} no está en el modelo metabólico: discuta qué implica eso para su análisis.")

**Análisis:**
- La reacción `G3PD5` (glicerol-3-fosfato deshidrogenasa aerobia, *glpD*) y `GLYCDx` (glicerol deshidrogenasa, *gldA*) representan las dos rutas de entrada. ¿Cómo cambia su expresión relativa entre REF, OPT y ALE?
- ¿Por qué `PDH` toma un valor más bajo que el de su gen más expresado?

## 🧩 Parte 3 — Modelos específicos de contexto con GIMME

### 3.1 ¿Cómo funciona GIMME?

**GIMME** (*Gene Inactivity Moderated by Metabolism and Expression*; Becker y Palsson, 2008, *PLoS Comput. Biol.*) busca la distribución de flujos que:

1. Cumple una **función metabólica requerida** (aquí: al menos una fracción del crecimiento máximo, `GIMME_FRACCION`, que le fue asignada), y
2. **Usa lo menos posible** las reacciones cuya expresión está por debajo de un umbral.

Matemáticamente resuelve un problema lineal:

$$\min \sum_{i:\,x_i<\theta} (\theta - x_i)\,|v_i| \quad \text{s.a.} \quad \mathbf{S}\cdot\mathbf{v}=0,\; lb \le v \le ub,\; v_{bio} \ge 0.9\,\mu_{max}$$

donde $x_i$ es la expresión de la reacción $i$ y $\theta$ el umbral. Cada reacción poco expresada recibe una penalización proporcional a "qué tan por debajo" del umbral está. El valor óptimo se llama **puntaje de inconsistencia**: cuánto tuvo que "contradecir" el modelo a los datos para cumplir la función requerida.

El **modelo de contexto** conserva: (i) las reacciones expresadas por encima del umbral, (ii) las reacciones sin genes asociados y (iii) las reacciones poco expresadas que GIMME necesitó usar.

Para el valor absoluto $|v_i|$ usamos las variables *forward* y *reverse* que COBRApy crea para cada reacción ($v = v^+ - v^-$, con $v^+, v^- \ge 0$).

**Preguntas de anticipación:**
- ¿Qué pasaría con un umbral muy alto? ¿Y con uno muy bajo?
- ¿Por qué exigir solo una fracción (p. ej. 80–95 %) del crecimiento máximo y no el 100 %? ¿Qué espera que cambie con su fracción asignada?

In [ ]:
GIMME_PERCENTIL = ASIGNACION["gimme_percentil"]   # umbral = percentil de la expresión por reacción (asignado)
GIMME_FRACCION  = ASIGNACION["gimme_fraccion"]    # fracción del crecimiento máximo que se exige (asignada)
print(f"GIMME con su asignación: percentil {GIMME_PERCENTIL}, fracción de crecimiento {GIMME_FRACCION:.2f}")
PARAMETROS["GIMME"] = {"umbral_percentil": GIMME_PERCENTIL, "fraccion_crecimiento": GIMME_FRACCION,
                       "expresion": "log2(TPM+1); GPR AND=min, OR=max"}

def gimme(model, expr, percentil=GIMME_PERCENTIL, fraccion=GIMME_FRACCION):
    # Devuelve (modelo_contexto, umbral, puntaje_inconsistencia, flujos_gimme)
    umbral = np.nanpercentile(expr.values, percentil)
    with model as m:
        mu_max = m.slim_optimize()
        m.reactions.get_by_id(BIOMASA).lower_bound = fraccion * mu_max
        coef = {}
        for r in m.reactions:
            x = expr.get(r.id, np.nan)
            if not np.isnan(x) and x < umbral:
                coef[r.forward_variable] = umbral - x
                coef[r.reverse_variable] = umbral - x
        m.objective = m.problem.Objective(0, direction="min")
        m.objective.set_linear_coefficients(coef)
        sol = m.optimize()
        if sol.status != "optimal":
            raise RuntimeError(f"❌ GIMME no encontró solución ({sol.status})")
        flujos = sol.fluxes.copy()
        inconsistencia = sol.objective_value
    conservar = {r.id for r in model.reactions
                 if np.isnan(expr.get(r.id, np.nan)) or expr[r.id] >= umbral or abs(flujos[r.id]) > 1e-7}
    ctx = model.copy()
    ctx.remove_reactions([r for r in ctx.reactions if r.id not in conservar], remove_orphans=True)
    return ctx, umbral, inconsistencia, flujos

modelos = {"Base": modelo}
filas = []
for c in CONDICIONES:
    t0 = time.time()
    ctx, umbral, inc, _ = gimme(modelo, expr_rxn[c])
    ctx.id = f"iEC1349_{c}"
    modelos[c] = ctx
    filas.append({"Modelo": ctx.id, "Condición": NOMBRE[c], "Umbral log2(TPM+1)": round(umbral, 2),
                  "Inconsistencia": round(inc, 2), "Reacciones": len(ctx.reactions),
                  "Metabolitos": len(ctx.metabolites), "Genes": len(ctx.genes),
                  "μ máx (1/h)": round(ctx.slim_optimize(), 3)})
    print(f"✅ {ctx.id} construido en {time.time()-t0:.1f} s")
tabla_modelos = pd.DataFrame(filas)
tabla_modelos.loc[len(tabla_modelos)] = {"Modelo": modelo.id, "Condición": "Modelo genérico", "Umbral log2(TPM+1)": np.nan,
    "Inconsistencia": np.nan, "Reacciones": len(modelo.reactions), "Metabolitos": len(modelo.metabolites),
    "Genes": len(modelo.genes), "μ máx (1/h)": round(modelo.slim_optimize(), 3)}
display(tabla_modelos)
guardar_tabla(tabla_modelos, "T4_modelos_contexto")

for c in CONDICIONES:
    write_sbml_model(modelos[c], f"{MOD}/iEC1349_{c}.xml")
print(f"✅ Modelos de contexto guardados en SBML en {MOD}")

**Análisis:**
- ¿Qué condición tiene el mayor puntaje de inconsistencia? ¿Qué significa biológicamente que el modelo tenga que usar muchas reacciones poco expresadas para crecer?
- Los tres modelos crecen casi igual. ¿Por qué? (Pista: revise la restricción de crecimiento que impusimos.) ¿Qué datos experimentales necesitaría para que el modelo prediga tasas de crecimiento diferentes?
- Compare con los modelos del artículo original (iTA818 e iTA821 tenían ≈ 800 genes). ¿Por qué nuestros modelos conservan más genes?

### 3.2 Figura 3 — Comparación de los modelos de contexto

- **A:** distribución de la expresión por reacción y el umbral de GIMME en cada condición.
- **B:** reacciones compartidas y exclusivas entre los tres modelos (diagrama tipo *UpSet*).
- **C:** sensibilidad del tamaño del modelo al percentil elegido como umbral.

**Preguntas antes de ejecutar:**
- ¿Espera que los modelos OPT y ALE se parezcan más entre sí que a REF? ¿Por qué?

In [ ]:
# Sensibilidad al umbral (panel C)
PERCENTILES = sorted({10, 25, 40, 50, GIMME_PERCENTIL})
sens = []
for c in CONDICIONES:
    for p in PERCENTILES:
        ctx_p, _, inc_p, _ = gimme(modelo, expr_rxn[c], percentil=p)
        sens.append({"Condición": c, "Percentil": p, "Reacciones": len(ctx_p.reactions), "Inconsistencia": inc_p})
sens = pd.DataFrame(sens)
guardar_tabla(sens, "S3_sensibilidad_umbral_GIMME")

# Conjuntos de reacciones (panel B)
conj = {c: {r.id for r in modelos[c].reactions} for c in CONDICIONES}
todas = set().union(*conj.values())
patrones = {}
for r in todas:
    clave = tuple(c for c in CONDICIONES if r in conj[c])
    patrones[clave] = patrones.get(clave, 0) + 1
patrones = sorted(patrones.items(), key=lambda kv: -kv[1])

# Máximo dos paneles por fila; el tercero va centrado abajo (9 cm de ancho)
fig = nueva_figura(columnas=2, alto_cm=13.0)
gs = fig.add_gridspec(2, 4)
# --- A
ax = fig.add_subplot(gs[0, 0:2])
for c in CONDICIONES:
    v = np.sort(expr_rxn[c].dropna().values)
    ax.plot(v, np.linspace(0, 1, len(v)), color=COLOR[c], label=c, lw=1)
    u = np.nanpercentile(expr_rxn[c].values, GIMME_PERCENTIL)
    ax.axvline(u, color=COLOR[c], lw=0.6, ls="--")
ax.set_xlabel("Expresión por reacción, log$_2$(TPM+1)"); ax.set_ylabel("Fracción acumulada de reacciones")
ax.set_title("Umbral de GIMME"); ax.legend(loc="lower right")
ax.text(0.03, 0.97, f"-- percentil {GIMME_PERCENTIL}", transform=ax.transAxes, va="top", fontsize=5.5, color="#52514e")
letra_panel(ax, "A")
# --- B (UpSet simplificado)
gs_b = gs[0, 2:4].subgridspec(2, 1, height_ratios=[3, 1.1])
axb = fig.add_subplot(gs_b[0]); axm = fig.add_subplot(gs_b[1], sharex=axb)
x = np.arange(len(patrones))
vals = [n for _, n in patrones]
axb.bar(x, vals, color="#52514e", width=0.6)
axb.set_yscale("log"); axb.set_ylabel("Reacciones")
for xi, v in zip(x, vals):
    axb.text(xi, v * 1.15, f"{v}", ha="center", fontsize=5)
axb.tick_params(axis="x", labelbottom=False, length=0); axb.set_title("Reacciones compartidas")
for j, c in enumerate(CONDICIONES):
    for xi, (clave, _) in enumerate(patrones):
        axm.scatter(xi, j, s=14, color=COLOR[c] if c in clave else "#e1e0d9", zorder=3)
    axm.text(-0.8, j, c, ha="right", va="center", fontsize=6)
for xi, (clave, _) in enumerate(patrones):
    idx = [CONDICIONES.index(c) for c in clave]
    axm.plot([xi, xi], [min(idx), max(idx)], color="#52514e", lw=0.8, zorder=2)
axm.set_ylim(-0.6, 2.6); axm.axis("off")
letra_panel(axb, "B")
# --- C
ax = fig.add_subplot(gs[1, 1:3])
for c in CONDICIONES:
    d = sens[sens["Condición"] == c]
    ax.plot(d["Percentil"], d["Reacciones"], "o-", color=COLOR[c], label=c, ms=3)
ax.axhline(len(modelo.reactions), color="#898781", lw=0.6, ls="--")
ax.text(PERCENTILES[0], len(modelo.reactions), " modelo genérico", va="bottom", fontsize=5.5, color="#52514e")
ax.set_xlabel("Percentil usado como umbral"); ax.set_ylabel("Reacciones en el modelo de contexto")
ax.set_title("Sensibilidad al umbral"); ax.legend(loc="lower left")
letra_panel(ax, "C")
guardar_figura(fig, "Figura3_modelos_contexto")

jac = pd.DataFrame({a: {b: len(conj[a] & conj[b]) / len(conj[a] | conj[b]) for b in CONDICIONES} for a in CONDICIONES})
print("Índice de Jaccard entre modelos:"); display(jac.round(3))

In [ ]:
# ¿Qué reacciones distinguen a cada modelo? (útil para la discusión)
filas = []
for c in CONDICIONES:
    otros = set().union(*(conj[o] for o in CONDICIONES if o != c))
    for rid in sorted(conj[c] - otros):
        r = modelo.reactions.get_by_id(rid)
        filas.append({"Modelo": c, "Tipo": "exclusiva", "Reacción": rid, "Nombre": r.name, "GPR": r.gene_reaction_rule})
for c in CONDICIONES:
    otros_ = [o for o in CONDICIONES if o != c]
    for rid in sorted((conj[otros_[0]] & conj[otros_[1]]) - conj[c]):
        r = modelo.reactions.get_by_id(rid)
        filas.append({"Modelo": c, "Tipo": "ausente (presente en los otros dos)", "Reacción": rid,
                      "Nombre": r.name, "GPR": r.gene_reaction_rule})
dif_rxn = pd.DataFrame(filas)
guardar_tabla(dif_rxn, "S4_reacciones_diferenciales_modelos")
display(dif_rxn.groupby(["Modelo", "Tipo"]).size().rename("n").to_frame())
display(dif_rxn[dif_rxn["Modelo"] == "REF"].head(15))

**Análisis — Figura 3:**
- ¿Cómo cambia el número de reacciones al aumentar el percentil? ¿Hay un punto a partir del cual el modelo "se rompe" o pierde muchas reacciones?
- Revise la tabla `S4_reacciones_diferenciales_modelos.csv`. Elija 3–5 reacciones exclusivas o ausentes de cada modelo y discuta si tienen sentido biológico (transporte de fosfato, rutas de nitrógeno, metabolismo del glicerol…).
- **Para el artículo:** discuta su percentil asignado frente a los demás valores del panel C. ¿Qué tan robustas son sus conclusiones a ese parámetro?

## 🌊 Parte 4 — Distribución de flujos: ¿qué rutas usa cada condición?

### 4.1 Figura 4 — pFBA en el metabolismo central

El FBA suele tener **muchas soluciones óptimas** equivalentes. El **FBA parsimonioso (pFBA)** elige, entre ellas, la que minimiza la suma total de flujos (la célula "ahorra" enzima). Compararemos los flujos del metabolismo central entre el modelo genérico y los tres modelos de contexto.

**Preguntas antes de ejecutar:**
- ¿Por qué no basta con comparar la tasa de crecimiento entre modelos?
- ¿Qué reacciones espera que cambien entre REF y ALE según la Figura 2?

In [ ]:
REACCIONES_CENTRALES = {
    "Entrada de glicerol": ["GLYCtpp", "GLYK", "G3PD5", "G3PD2", "GLYCDx", "DHAPT", "F6PA"],
    "Glucólisis":          ["TPI", "FBA", "FBP", "GAPD", "PGK", "ENO", "PYK", "PPS"],
    "Pentosas fosfato":    ["G6PDH2r", "GND", "TKT1", "TALA"],
    "Nodo PEP/piruvato":   ["PDH", "PFL", "PPC", "PPCK", "ME1", "ME2"],
    "TCA y glioxilato":    ["CS", "ACONTa", "ICDHyr", "AKGDH", "SUCOAS", "SUCDi", "FUM", "MDH", "ICL", "MALS"],
    "Rama reductiva":      ["FRD2", "FRD3"],
    "Productos":           ["PTAr", "ACKr", "POX", "LDH_D", "EX_ac_e", "EX_succ_e", "EX_co2_e", "EX_o2_e"],
}
lista_rxn, ruta_rxn = [], []
for g, rs in REACCIONES_CENTRALES.items():
    rs = [r for r in rs if r in modelo.reactions]
    lista_rxn += rs; ruta_rxn += [g] * len(rs)

ORDEN_MOD = ["Base", "REF", "OPT", "ALE"]
flujos = {}
for k in ORDEN_MOD:
    s = pfba(modelos[k])
    flujos[k] = s.fluxes
F = pd.DataFrame({k: flujos[k].reindex(lista_rxn) for k in ORDEN_MOD}).fillna(0.0)
F_tabla = F.copy(); F_tabla.insert(0, "Ruta", ruta_rxn)
guardar_tabla(F_tabla.round(4).reset_index().rename(columns={"index": "Reacción"}), "T5_flujos_pFBA_centrales")
todas_pfba = pd.DataFrame(flujos).round(6)
guardar_tabla(todas_pfba.reset_index().rename(columns={"index": "Reacción"}), "S5_flujos_pFBA_todas")

# Solo se grafican las reacciones con flujo en al menos un modelo
activas = F.index[(F.abs() > 1e-6).any(axis=1)]
Fp = F.loc[activas]
rutas_p = [ruta_rxn[lista_rxn.index(r)] for r in activas]

fig = nueva_figura(columnas=1, alto_cm=12.5)
ax = fig.subplots()
lim = np.ceil(np.nanpercentile(np.abs(Fp.values), 98))
im = ax.imshow(Fp.values, aspect="auto", cmap=DIVERGENTE, norm=TwoSlopeNorm(0, -lim, lim))
ax.set_xticks(range(len(ORDEN_MOD)), ["Genérico" if k == "Base" else k for k in ORDEN_MOD])
ax.xaxis.tick_top()
ax.set_yticks(range(len(activas)), activas)
for i, r in enumerate(activas):
    for j, k in enumerate(ORDEN_MOD):
        v = Fp.loc[r, k]
        if abs(v) > 1e-6:
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=5,
                    color="white" if abs(v) > lim * 0.6 else "#0b0b0b")
inicio = 0
for g in REACCIONES_CENTRALES:
    n = rutas_p.count(g)
    if n == 0:
        continue
    if inicio > 0:
        ax.axhline(inicio - 0.5, color="white", lw=2)
    ax.text(len(ORDEN_MOD) - 0.4, inicio + n / 2 - 0.5, g, ha="left", va="center", fontsize=5.5, color="#52514e")
    inicio += n
ax.spines[:].set_visible(False); ax.tick_params(length=0)
cb = fig.colorbar(im, ax=ax, orientation="horizontal", shrink=0.6, pad=0.01, aspect=30, extend="both")
cb.set_label("Flujo pFBA (mmol/gDW/h)"); cb.outline.set_visible(False)
guardar_figura(fig, "Figura4_flujos_pFBA")

**Análisis — Figura 4:**
- ¿Qué fracción del glicerol entra por `GLYK` (glicerol quinasa) y cuál por `GLYCDx` → `DHAPT` (glicerol deshidrogenasa → dihidroxiacetona quinasa) en cada modelo? ¿Concuerda con la expresión de *glpK* y *gldA* de la Figura 2?
- El modelo oxida el glicerol-3-fosfato con `G3PD2` (NADP⁺) y no con `G3PD5` (*glpD*, quinona), aunque *glpD* es uno de los genes más expresados. ¿Por qué el pFBA "prefiere" esa ruta? ¿Es un artefacto del modelo? ¿Cómo podría corregirse?
- ¿Algún modelo secreta succinato o acetato en el óptimo? ¿Qué le dice eso sobre la necesidad de ingeniería metabólica?

### 4.2 Figura 5 — Análisis de variabilidad de flujos (FVA)

El pFBA da **una** solución; el **FVA** calcula, para cada reacción, el flujo mínimo y máximo compatibles con un crecimiento ≥ 90 % del óptimo. Si el rango de una reacción se reduce en un modelo de contexto, los datos de expresión están **restringiendo** el metabolismo.

In [ ]:
FVA_FRACCION = 0.9
PARAMETROS["FVA"] = {"fraccion_optimo": FVA_FRACCION}
RXN_FVA = ["GLYK", "GLYCDx", "DHAPT", "G3PD5", "G3PD2", "F6PA", "PPC", "PPCK", "ME1", "ME2", "PDH", "PFL",
           "ICL", "MALS", "SUCDi", "FUM", "FRD2", "PTAr", "POX", "EX_ac_e", "EX_succ_e"]
RXN_FVA = [r for r in RXN_FVA if r in modelo.reactions]

fva = {}
for k in ORDEN_MOD:
    presentes = [r for r in RXN_FVA if r in modelos[k].reactions]
    fva[k] = flux_variability_analysis(modelos[k], reaction_list=presentes, fraction_of_optimum=FVA_FRACCION)
fva_tabla = pd.concat({k: v for k, v in fva.items()}, names=["Modelo", "Reacción"]).round(4)
guardar_tabla(fva_tabla.reset_index(), "T6_FVA_reacciones_clave")

fig = nueva_figura(columnas=2, alto_cm=9.0)
ax = fig.subplots()
desp = {"Base": -0.27, "REF": -0.09, "OPT": 0.09, "ALE": 0.27}
for k in ORDEN_MOD:
    for j, r in enumerate(RXN_FVA):
        if r in fva[k].index:
            lo, hi = fva[k].loc[r, "minimum"], fva[k].loc[r, "maximum"]
            ax.plot([lo, hi], [j + desp[k]] * 2, color=COLOR[k], lw=1.6, solid_capstyle="round")
        else:
            ax.plot(0, j + desp[k], marker="x", color=COLOR[k], ms=3, mew=0.8)
for k in ORDEN_MOD:
    ax.plot([], [], color=COLOR[k], lw=1.6, label=NOMBRE[k])
ax.plot([], [], "x", color="#52514e", ms=3, label="Reacción eliminada por GIMME")
ax.set_yticks(range(len(RXN_FVA)), RXN_FVA); ax.invert_yaxis()
ax.axvline(0, color="#898781", lw=0.5)
ax.set_xscale("symlog", linthresh=1)
ax.set_xlabel("Rango de flujo (mmol/gDW/h, escala symlog)")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=5, columnspacing=1.0, handlelength=1.5)
ax.grid(axis="x", color="#e1e0d9", lw=0.4)
guardar_figura(fig, "Figura5_FVA")

**Análisis — Figura 5:**
- ¿Qué reacciones tienen un rango mucho menor en los modelos de contexto que en el genérico?
- ¿Cuál es el máximo de succinato (`EX_succ_e`) que cada modelo puede secretar manteniendo el 90 % del crecimiento? ¿Qué condición tiene mayor "potencial" de producción?
- Si una reacción conserva un rango amplio en todos los modelos, ¿qué le dice eso sobre los datos de transcriptómica para esa reacción?

### 4.3 🔎 Su reacción asignada

Resumen de su reacción asignada: expresión, presencia en cada modelo, flujo pFBA y rango FVA. **Debe discutirla en su artículo**: ¿en qué ruta está?, ¿cambia entre condiciones?, ¿la expresión explica el flujo predicho?

In [ ]:
RXN = ASIGNACION["reaccion_asignada"]
r = modelo.reactions.get_by_id(RXN)
print(f"🔎 Reacción asignada: {RXN} — {r.name}")
print(f"   Ecuación: {r.reaction}")
print(f"   GPR:      {r.gene_reaction_rule}")
filas = []
for k in ORDEN_MOD:
    presente = RXN in modelos[k].reactions
    with modelos[k] as m:
        rango = (flux_variability_analysis(m, [RXN], fraction_of_optimum=FVA_FRACCION).iloc[0]
                 if presente else pd.Series({"minimum": np.nan, "maximum": np.nan}))
    filas.append({"Modelo": k, "En el modelo": "sí" if presente else "no (eliminada por GIMME)",
                  "Expresión log2(TPM+1)": round(expr_rxn.loc[RXN, k], 2) if k != "Base" and not np.isnan(expr_rxn.loc[RXN, k]) else "—",
                  "Flujo pFBA": round(flujos[k].get(RXN, 0.0), 3),
                  "FVA mín": round(rango["minimum"], 3), "FVA máx": round(rango["maximum"], 3)})
rxn_tabla = pd.DataFrame(filas)
display(rxn_tabla)
guardar_tabla(rxn_tabla, f"P2_reaccion_asignada_{RXN}")

## 🏭 Parte 5 — Diseño de cepas para sobreproducir ácido succínico

### 5.1 Figura 6 — Espacio de producción y efecto del oxígeno

La **envolvente de producción** muestra todas las combinaciones factibles de crecimiento y producción de succinato. El panel B explora cómo cambia el máximo de succinato al limitar el oxígeno, manteniendo el 50 % del crecimiento.

**Preguntas antes de ejecutar:**
- ¿Por qué el succinato se acumula más en condiciones de oxígeno limitado?
- ¿Qué tan cerca del borde superior de la envolvente cree que opera una célula silvestre?

In [ ]:
fig = nueva_figura(columnas=2, alto_cm=6.5)
axs = fig.subplots(1, 2)
def envolvente(m, puntos=30):
    # Para cada tasa de crecimiento: succinato mínimo y máximo factibles
    return production_envelope(m, [BIOMASA], objective="EX_succ_e", carbon_sources="EX_glyc_e", points=puntos)

env = {}
for k in ORDEN_MOD:
    e = envolvente(modelos[k])
    env[k] = e
    estilo = dict(color=COLOR[k], lw=1.4 if k != "Base" else 1.0, ls="-" if k != "Base" else "--")
    axs[0].plot(e[BIOMASA], e["flux_maximum"], label=NOMBRE[k], **estilo)
    axs[0].plot(e[BIOMASA], e["flux_minimum"], **estilo)
axs[0].set_xlabel("Tasa de crecimiento (1/h)"); axs[0].set_ylabel("Producción de succinato (mmol/gDW/h)")
axs[0].set_title("Envolvente de producción"); axs[0].legend(loc="upper right")
axs[0].set_xlim(left=0); axs[0].set_ylim(bottom=0)
guardar_tabla(pd.concat(env, names=["Modelo", "i"]).reset_index(), "S6_envolventes_produccion")

O2 = np.linspace(0, 20, 21)
o2_res = []
for k in ORDEN_MOD:
    for o in O2:
        with modelos[k] as m:
            m.reactions.EX_o2_e.lower_bound = -o
            mu = m.slim_optimize(error_value=0.0)
            succ = 0.0
            if mu > 1e-6:
                m.reactions.get_by_id(BIOMASA).lower_bound = 0.5 * mu
                m.objective = "EX_succ_e"
                succ = m.slim_optimize(error_value=0.0)
        o2_res.append({"Modelo": k, "O2_max": o, "mu_max": mu, "succ_max_50pct": succ})
o2_res = pd.DataFrame(o2_res)
guardar_tabla(o2_res, "S7_barrido_oxigeno")
for k in ORDEN_MOD:
    d = o2_res[o2_res["Modelo"] == k]
    axs[1].plot(d["O2_max"], d["succ_max_50pct"], marker="o", ms=2.5, color=COLOR[k], label=NOMBRE[k],
                ls="-" if k != "Base" else "--")
axs[1].set_xlabel("Captación máxima de O$_2$ (mmol/gDW/h)")
axs[1].set_ylabel("Succinato máx. con μ ≥ 50 % μ$_{máx}$")
axs[1].set_title("Efecto del oxígeno"); axs[1].legend(loc="upper right")
for a, l in zip(axs, "AB"):
    letra_panel(a, l)
guardar_figura(fig, "Figura6_envolvente_oxigeno")

**Análisis — Figura 6:**
- Describa la forma de la envolvente: ¿hay un compromiso (*trade-off*) entre crecer y producir succinato? ¿Cuál es el succinato **mínimo** en la cepa silvestre a cualquier tasa de crecimiento? ¿Qué significa?
- ¿Las envolventes de los modelos de contexto son iguales a la del modelo genérico? Si difieren, ¿qué reacciones eliminadas por GIMME lo explican?
- ¿A qué nivel de oxígeno se maximiza el succinato? ¿Recomendaría operar el biorreactor en aerobiosis, microaerobiosis o anaerobiosis? Justifique.

### 5.2 Búsqueda combinatoria de *knockouts* acoplados al crecimiento

El artículo original usó **OptKnock**, que resuelve un problema de optimización binivel (MILP) y requiere solvers comerciales para modelos grandes. Aquí haremos una **búsqueda exhaustiva** sobre las reacciones candidatas que OptKnock predijo con mayor frecuencia en ese trabajo (más algunas de control), evaluando todas las combinaciones de 1 a `MAX_KO` deleciones.

Para cada mutante calculamos:
- **μ**: crecimiento máximo del mutante.
- **Succinato mínimo a μ máximo**: el succinato que la célula está **obligada** a producir cuando crece al máximo. Si es > 0, la producción está **acoplada al crecimiento**: una cepa que evoluciona para crecer más, produce más.
- **BPCY** (*biomass-product coupled yield*) = μ × succinato / glicerol: un índice que combina productividad y crecimiento.

Repetimos la búsqueda en dos escenarios: **aerobio** (sin límite de O₂) y **O₂ limitado** (con el límite de captación que le fue asignado, `ASIGNACION["o2_limitado"]`), porque la Figura 6B muestra que el oxígeno cambia por completo el espacio de producción.

⏱️ Con `MAX_KO = 3` toma ≈ 2–3 minutos en total en Colab. Con `MAX_KO = 4`, ≈ 6–8 minutos.

In [ ]:
CANDIDATAS = ["ACKr", "PTAr", "PFL", "PDH", "FUM", "SUCDi", "TPI", "G3PD2", "GLYCDx", "F6PA",
              "POX", "LDH_D", "G6PDH2r", "PGL", "ICDHyr"]
MAX_KO = 3
ESCENARIOS_O2 = {"Aerobio": 1000, "O2 limitado": ASIGNACION["o2_limitado"]}   # captación máx. de O2 (mmol/gDW/h)
PARAMETROS["knockouts"] = {"candidatas": CANDIDATAS, "max_knockouts": MAX_KO, "escenarios_O2": ESCENARIOS_O2,
                           "criterio": "succinato mínimo a μ máximo (acoplamiento al crecimiento)"}

def evaluar_mutante(m, kos):
    with m:
        for k in kos:
            m.reactions.get_by_id(k).knock_out()
        mu = m.slim_optimize(error_value=0.0)
        if mu < 1e-3:
            return mu, 0.0, 0.0
        m.reactions.get_by_id(BIOMASA).lower_bound = 0.999 * mu
        m.objective = "EX_succ_e"
        m.objective_direction = "max"
        s_max = m.slim_optimize(error_value=0.0)
        m.objective_direction = "min"
        s_min = m.slim_optimize(error_value=0.0)
        return mu, s_min, s_max

ko_res, mu_wt = [], {}
for esc, o2 in ESCENARIOS_O2.items():
    for k in CONDICIONES:
        m = modelos[k]
        presentes = [c for c in CANDIDATAS if c in m.reactions]
        t0 = time.time()
        with m:
            m.reactions.EX_o2_e.lower_bound = -o2
            mu_wt[(esc, k)] = m.slim_optimize()
            for n in range(1, MAX_KO + 1):
                for kos in itertools.combinations(presentes, n):
                    mu, s_min, s_max = evaluar_mutante(m, kos)
                    ko_res.append({"Escenario": esc, "Modelo": k, "Knockouts": " + ".join(kos), "n_KO": n,
                                   "mu": mu, "succ_min": s_min, "succ_max": s_max})
        print(f"✅ {esc:12s} | {k}: mutantes evaluados en {time.time()-t0:.0f} s "
              f"(μ silvestre = {mu_wt[(esc, k)]:.3f} 1/h)")

ko_res = pd.DataFrame(ko_res)
ko_res["mu_relativo"] = ko_res.apply(lambda r: r["mu"] / mu_wt[(r["Escenario"], r["Modelo"])], axis=1)
ko_res["Rendimiento (mol succ/mol glyc)"] = ko_res["succ_min"] / GLICEROL_UPTAKE
ko_res["BPCY"] = ko_res["mu"] * ko_res["succ_min"] / GLICEROL_UPTAKE
guardar_tabla(ko_res.round(5), "S8_knockouts_todos")

# Acoplado = producción obligatoria > 0 y crecimiento > 10 % del silvestre
acoplados = ko_res[(ko_res["succ_min"] > 1e-3) & (ko_res["mu_relativo"] > 0.1)].copy()
# Estrategia MÍNIMA: ninguna de sus deleciones sobra. Si al quitar una deleción el mutante
# sigue produciendo lo mismo, esa deleción no aporta y la estrategia no es mínima.
succ_por_conjunto = {(r["Escenario"], r["Modelo"], frozenset(r["Knockouts"].split(" + "))): r["succ_min"]
                     for _, r in ko_res.iterrows()}
def es_minima(r):
    kos = frozenset(r["Knockouts"].split(" + "))
    if len(kos) == 1:
        return True
    return all(succ_por_conjunto.get((r["Escenario"], r["Modelo"], kos - {x}), 0.0) < 0.99 * r["succ_min"]
               for x in kos)
minimas = acoplados[acoplados.apply(es_minima, axis=1)].copy()
# Entre mutantes empatados se prefiere el que tiene MENOS deleciones (más fácil de construir);
# si siguen empatados, se ordenan alfabéticamente para que el resultado sea reproducible.
for df_ in (ko_res, acoplados, minimas):
    df_["succ_orden"] = df_["succ_min"].round(4)
    df_["mu_orden"] = df_["mu"].round(4)
ORDEN_MEJOR = dict(by=["succ_orden", "n_KO", "mu_orden", "Knockouts"], ascending=[False, True, False, True])
top = (minimas.sort_values(**ORDEN_MEJOR).groupby(["Escenario", "Modelo"]).head(5)
       .sort_values(["Escenario", "Modelo"]))
guardar_tabla(minimas.sort_values(["Escenario", "Modelo", "succ_min"], ascending=[True, True, False]).round(4),
              "T7_estrategias_minimas")
print("\nMutantes acoplados al crecimiento (todos / estrategias mínimas):")
display(pd.concat({"acoplados": acoplados.groupby(["Escenario", "Modelo"]).size(),
                   "mínimas": minimas.groupby(["Escenario", "Modelo"]).size()}, axis=1))
print("Las 5 mejores estrategias mínimas por modelo y escenario:")
display(top.round(3))

### 5.3 Figura 7 — Estrategias de ingeniería metabólica por modelo

- **A–C:** todos los mutantes de REF, OPT y ALE en el plano crecimiento–succinato (círculos: aerobio; triángulos: O₂ limitado). Los puntos por encima de cero están acoplados al crecimiento.
- **D:** succinato **obligatorio** (el mínimo que la célula debe secretar) a cada tasa de crecimiento, para la mejor estrategia mínima de cada modelo con su O₂ limitado asignado (línea continua; el punto marca el crecimiento máximo del mutante) frente a su silvestre *in silico* (línea discontinua).
- **E:** frecuencia con que cada reacción aparece entre las **estrategias mínimas** acopladas de cada modelo y escenario (comparable a la Figura 5 del artículo original).

> **Estrategia mínima:** combinación de *knockouts* en la que todas las deleciones son necesarias; si se quita cualquiera, la producción obligatoria de succinato cae. La búsqueda exhaustiva evalúa todas las combinaciones, así que sin este filtro las deleciones "acompañantes" (que no aportan nada) inflarían las frecuencias.

In [ ]:
# Filas 1–2: A–C (un modelo por panel) y D (acoplamiento); fila 3: E (heatmap) de lado a lado
fig = nueva_figura(columnas=2, alto_cm=21.0)
gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 0.9])
POS = {"REF": gs[0, 0], "OPT": gs[0, 1], "ALE": gs[1, 0]}
MARCA = {"Aerobio": "o", "O2 limitado": "^"}
# --- A, B, C: un panel por modelo
ejes_a = []
for i, k in enumerate(CONDICIONES):
    ax = fig.add_subplot(POS[k], sharey=ejes_a[0] if ejes_a else None)
    ejes_a.append(ax)
    for esc in ESCENARIOS_O2:
        d = ko_res[(ko_res["Modelo"] == k) & (ko_res["Escenario"] == esc)]
        ax.scatter(d["mu"], d["succ_min"], s=8, marker=MARCA[esc], lw=0.5, edgecolor=COLOR[k],
                   facecolor="none" if esc == "Aerobio" else COLOR[k], alpha=0.6, label=esc.replace("O2", "O$_2$"))
        mejor = minimas[(minimas["Modelo"] == k) & (minimas["Escenario"] == esc)].sort_values(**ORDEN_MEJOR)
        if len(mejor):
            b = mejor.iloc[0]
            ax.annotate("Δ" + b["Knockouts"].replace(" + ", "\nΔ"), (b["mu"], b["succ_min"]),
                        xytext=(4, 0), textcoords="offset points", ha="left", va="center", fontsize=5)
    ax.set_xlabel("Crecimiento del mutante (1/h)")
    ax.set_ylabel("Succinato mínimo a μ máx. (mmol/gDW/h)")
    letra_panel(ax, "ABC"[i])
    ax.set_title(NOMBRE[k])
    if i == 0:
        ax.legend(loc="center left", bbox_to_anchor=(0.0, 0.62), handletextpad=0.1)
    ax.set_xlim(0, None)
ejes_a[0].set_ylim(-0.05, ko_res["succ_min"].max() * 1.25)
# --- E: frecuencia (heatmap secuencial), fila completa
ax = fig.add_subplot(gs[2, :])
frec = {}
for esc in ESCENARIOS_O2:
    for k in CONDICIONES:
        d = minimas[(minimas["Modelo"] == k) & (minimas["Escenario"] == esc)]
        cuenta = pd.Series([x for kos in d["Knockouts"] for x in kos.split(" + ")], dtype=object).value_counts()
        frec[f"{k} · {esc.replace('O2', 'O$_2$')} (n={len(d)})"] = (100 * cuenta / max(len(d), 1)).reindex(CANDIDATAS).fillna(0)
frec = pd.DataFrame(frec)
frec = frec.loc[frec.mean(axis=1).sort_values(ascending=False).index]
frec = frec.loc[frec.max(axis=1) > 0]
SECUENCIAL = LinearSegmentedColormap.from_list("seq", ["#f0efec", "#cde2fb", "#6da7ec", "#2a78d6", "#184f95", "#0d366b"])
im = ax.imshow(frec.T.values, aspect="auto", cmap=SECUENCIAL, vmin=0, vmax=100)
ax.set_xticks(range(len(frec)), frec.index, rotation=45, ha="right")
ax.set_yticks(range(frec.shape[1]), frec.columns, fontsize=5.5)
for i in range(frec.shape[1]):
    for j in range(frec.shape[0]):
        v = frec.iloc[j, i]
        if v >= 1:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=5, color="white" if v > 55 else "#0b0b0b")
ax.axhline(2.5, color="white", lw=2)
ax.spines[:].set_visible(False); ax.tick_params(length=0)
cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.01); cb.outline.set_visible(False)
cb.set_label("% de estrategias mínimas\ncon la deleción")
ax.set_title("Frecuencia en estrategias mínimas")
letra_panel(ax, "E")
guardar_tabla(frec.round(2).reset_index().rename(columns={"index": "Reacción"}), "S9_frecuencia_knockouts")
# --- D: succinato OBLIGATORIO (mínimo) vs crecimiento, con O2 limitado
ax = fig.add_subplot(gs[1, 1])
esc = "O2 limitado"
y_max = 0.0
for k in CONDICIONES:
    with modelos[k] as m:
        m.reactions.EX_o2_e.lower_bound = -ESCENARIOS_O2[esc]
        e_wt = envolvente(m, puntos=60)
        ax.plot(e_wt[BIOMASA], e_wt["flux_minimum"], color=COLOR[k], lw=0.8, ls="--")
        d = minimas[(minimas["Modelo"] == k) & (minimas["Escenario"] == esc)].sort_values(**ORDEN_MEJOR)
        if len(d):
            kos = d.iloc[0]["Knockouts"].split(" + ")
            for r in kos:
                m.reactions.get_by_id(r).knock_out()
            e = envolvente(m, puntos=60)
            ax.plot(e[BIOMASA], e["flux_minimum"], color=COLOR[k], lw=1.4, label=f"{k}: Δ{' Δ'.join(kos)}")
            ax.plot(e[BIOMASA].iloc[-1], e["flux_minimum"].iloc[-1], "o", ms=4, color=COLOR[k])
            y_max = max(y_max, e["flux_minimum"].max())
ax.plot([], [], color="#52514e", lw=0.8, ls="--", label="Silvestre in silico")
ax.set_xlabel("Tasa de crecimiento (1/h)"); ax.set_ylabel("Succinato obligatorio (mmol/gDW/h)")
ax.set_title(f"Acoplamiento (O$_2$ ≤ {ESCENARIOS_O2[esc]})"); ax.legend(loc="upper left", fontsize=5)
ax.set_xlim(left=0); ax.set_ylim(-0.05, max(y_max * 1.6, 0.5))
letra_panel(ax, "D")
guardar_figura(fig, "Figura7_knockouts")

**Análisis — Figuras 6 y 7:**
- En aerobiosis, ¿qué *knockout* aparece en casi todos los mutantes acoplados? ¿Por qué eliminar la fumarasa obligaría a la célula a secretar succinato? (Pista: siga el carbono en el ciclo de Krebs.)
- Con O₂ limitado aparecen otras deleciones (PDH, PFL…). ¿Por qué bloquear la conversión de piruvato en acetil-CoA desvía carbono hacia succinato? Compare con el artículo original, donde FUM y PDH fueron las más frecuentes en los tres modelos.
- ¿Las mejores estrategias son las mismas en REF, OPT y ALE? Si cambian, ¿qué reacciones de los modelos de contexto explican la diferencia?
- En el panel D, ¿a partir de qué tasa de crecimiento el mutante está obligado a producir succinato? ¿Por qué el acoplamiento es más fuerte justo en el crecimiento máximo? ¿Qué pasaría en un experimento de evolución adaptativa con esa cepa?
- Las deleciones sobre reacciones con isoenzimas (p. ej. `FUM`: *fumA*, *fumB*, *fumC*) requieren eliminar **varios genes**. ¿Cuántos genes habría que borrar en la práctica para su mejor mutante? Revise la columna GPR del modelo.
- El rendimiento teórico máximo de succinato desde glicerol es 1 mol/mol. ¿Qué tan cerca está su mejor mutante?

## 🧪 Parte 6 — Retos (para enriquecer el artículo)

Elija **al menos dos** retos e incluya sus resultados como figura o tabla suplementaria.

**Reto 1 — Robustez del método.** Repita la Parte 3 con `GIMME_PERCENTIL = 10` y `50`. ¿Cambian las reacciones más frecuentes en los mutantes acoplados?

**Reto 2 — Nivel de oxígeno.** Agregue a `ESCENARIOS_O2` los escenarios `"Anaerobio": 0` y `"Microaerobio": 2`. ¿Aparecen estrategias nuevas? ¿En qué nivel de oxígeno se obtiene el mejor compromiso entre crecimiento y rendimiento?

**Reto 3 — Más candidatos.** Agregue a `CANDIDATAS` reacciones que usted proponga a partir de la Figura 4 (por ejemplo `PPCK`, `ME1`, `ME2`, `MALS`, `ICL`, `SUCOAS`) y use `MAX_KO = 4`. ¿Encuentra mutantes mejores que los del artículo original?

**Reto 4 — Sobreexpresión.** La enzima PEP carboxilasa (`PPC`) es un blanco clásico de sobreexpresión para succinato. Fije un flujo mínimo en `PPC` (p. ej. `lower_bound = 5`) en el mejor mutante. ¿Mejora la producción?

**Reto 5 — Restricciones con la expresión diferencial.** En lugar de GIMME, implemente una versión simple de **E-Flux** (Colijn *et al.*, 2009): escale el límite superior de cada reacción proporcionalmente a su expresión relativa (TPM_condición / TPM_máximo). Compare las predicciones con las de GIMME.

In [ ]:
# Su código aquí (Reto 1):

In [ ]:
# Su código aquí (Reto 2):

In [ ]:
# Su código aquí (Reto 3, 4 o 5):

## ❓ Preguntas integradoras (para la Discusión del artículo)

**Sobre los datos**
1. ¿Qué intervención —el medio o la evolución— reprograma más el metabolismo central? Use las Figuras 1–3 y la Tabla 3 para argumentar.
2. Los genes del regulón *glp* se inducen en OPT y ALE. ¿Es suficiente la expresión para explicar que el medio optimizado consuma ≈ 30 g/L de glicerol frente a ≈ 7 g/L del medio base? ¿Qué papel podrían tener el nitrógeno (NH₄Cl) y la capacidad buffer (fosfatos) del medio?

**Sobre el método**
3. Los tres modelos de contexto predicen casi el mismo crecimiento. ¿Es un defecto del método o una consecuencia de cómo lo formulamos? ¿Cómo incorporaría tasas experimentales (consumo de glicerol, crecimiento) para obtener predicciones cuantitativas distintas?
4. ¿Qué supuestos introduce (a) usar TPM en lugar de conteos normalizados por DESeq2, (b) asignar 0 a los genes filtrados en REF, (c) usar el mínimo/máximo en las reglas GPR, (d) un medio mínimo en lugar de uno con extracto de levadura?
5. GIMME es un método "de umbral". ¿Qué ventajas y desventajas tiene frente a métodos continuos como E-Flux o métodos discretos como iMAT?

**Sobre su asignación personal**
- ¿Cómo cambiaron sus resultados (tamaño de los modelos, estrategias de *knockout*) por su percentil, su fracción de crecimiento y su nivel de O₂? Compare con al menos un compañero que tenga parámetros distintos.
- ¿Qué papel tiene su gen asignado en el paso de glicerol a succinato? ¿Su reacción asignada cambia de flujo entre REF, OPT y ALE? ¿Por qué?

**Sobre el diseño de la biofábrica**
6. ¿Las estrategias predichas son coherentes con las del artículo original (FUM, PDH, PFL, PTAr, ACKr…)? ¿Qué explica las diferencias (modelo iEC1349 vs iTA1338, búsqueda exhaustiva vs OptKnock, umbral de GIMME)?
7. ¿Qué cepa de partida (silvestre en medio optimizado o evolucionada) elegiría para implementar los *knockouts*? Justifique con sus resultados.
8. Proponga un experimento concreto (cepa, medio, condiciones de aireación, mediciones) para validar su mejor predicción.

> **Recuerde:** los modelos predicen; los experimentos validan.

## 🗂️ Guía para escribir el artículo

| Sección | Qué incluir | Resultados del cuaderno |
|:--|:--|:--|
| **Métodos 2.1** Datos de RNA-seq | Condiciones, réplicas, criterio de DEG | Tabla de casos, `T1_resumen_DEG.csv` |
| **Métodos 2.2** Modelo y medio | iEC1349_Crooks, medio mínimo, 13.3 mmol/gDW/h de glicerol | `metodos_parametros.txt` |
| **Métodos 2.3** Integración | TPM, GPR (min/max), GIMME (su percentil y fracción asignados) | `metodos_parametros.txt` |
| **Métodos 2.4** Simulaciones | pFBA, FVA, envolventes, búsqueda de *knockouts* | `metodos_parametros.txt` |
| **Resultados 3.1** | El transcriptoma responde al medio y a la ALE | Figuras 1–2, Tablas T1–T3 |
| **Resultados 3.2** | Modelos de contexto | Figura 3, Tabla T4, S3–S4 |
| **Resultados 3.3** | Redistribución de flujos | Figuras 4–5, Tablas T5–T6 |
| **Resultados 3.4** | Estrategias para succinato | Figuras 6–7, Tabla T7 (estrategias mínimas), S8–S9 |
| **Discusión** | Interpretación, comparación con la literatura, limitaciones; **su gen y su reacción asignados** | Preguntas integradoras, tablas P1–P2 |
| **Disponibilidad de datos** | Código de estudiante y **código de verificación** | Se imprime al final del cuaderno |

**Checklist de figuras:** cada figura tiene leyenda (*caption*) autocontenida, define abreviaturas, indica *n* y umbrales estadísticos, y se cita en el texto en orden.

**Referencias clave:**
- Tafur Rangel AE *et al.* (2021) *In silico* design for systems-based metabolic engineering for the bioconversion of valuable compounds from industrial by-products. *Front. Genet.* 12:633073.
- Becker SA, Palsson BØ (2008) Context-specific metabolic networks are consistent with experiments. *PLoS Comput. Biol.* 4:e1000082.
- Ebrahim A *et al.* (2013) COBRApy: COnstraints-Based Reconstruction and Analysis for Python. *BMC Syst. Biol.* 7:74.
- Lewis NE *et al.* (2010) Omic data from evolved *E. coli* are consistent with computed optimal growth from genome-scale models. *Mol. Syst. Biol.* 6:390.
- Burgard AP, Pharkya P, Maranas CD (2003) OptKnock. *Biotechnol. Bioeng.* 84:647–657.
- Monk JM *et al.* (2016) Multi-omics quantification of species variation of *Escherichia coli* links molecular features with strain phenotypes. *Cell Syst.* 3:238–251.

## 📦 Paso final — Registrar parámetros, comprimir y descargar los resultados

Esta celda escribe `metodos_parametros.txt` (versiones de software y todos los parámetros usados, para la sección de Métodos), comprime la carpeta `resultados/` y la descarga.

In [ ]:
# Archivo de métodos
with open(f"{RES}/metodos_parametros.txt", "w", encoding="utf-8") as f:
    f.write(f"Taller Módulo 10 — generado el {datetime.datetime.now():%Y-%m-%d %H:%M}\n")
    f.write(f"Estudiante: {CODIGO_ESTUDIANTE} | Código de verificación: {CODIGO_VERIFICACION}\n\n")
    f.write("SOFTWARE\n")
    for k, v in VERSIONES.items():
        f.write(f"  {k}: {v}\n")
    f.write(f"  Plataforma: {platform.platform()}\n\n")
    f.write("PARÁMETROS\n")
    f.write(json.dumps(PARAMETROS, indent=2, ensure_ascii=False, default=str))
    f.write("\n\nMODELOS DE CONTEXTO\n")
    f.write(tabla_modelos.to_string(index=False))
print(open(f"{RES}/metodos_parametros.txt", encoding="utf-8").read())

# Comprimir todo
zip_path = f"{WORKDIR}/resultados_taller_m10_{CODIGO_ESTUDIANTE}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for ruta in glob.glob(f"{RES}/**/*", recursive=True):
        if os.path.isfile(ruta):
            z.write(ruta, os.path.relpath(ruta, WORKDIR))
print(f"\n✅ Resultados comprimidos en: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")
print(f"   Figuras: {len(glob.glob(f'{FIG}/*.pdf'))} | Tablas: {len(glob.glob(f'{TAB}/*.csv'))} | "
      f"Modelos: {len(glob.glob(f'{MOD}/*.xml'))}")

print(f"\n🔑 Código de verificación: {CODIGO_VERIFICACION}  →  inclúyalo en la sección 'Disponibilidad de datos'")
if EN_COLAB:
    from google.colab import files
    files.download(zip_path)
else:
    print("Descargue el archivo desde el explorador de archivos.")